<a href="https://colab.research.google.com/github/loudamn78-gif/thakaa-flow-training/blob/main/week2/day12/day12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 1

### 1. Feature Engineering و Feature Selection
مثالي من كرة القدم. الـ Target هو: هل يفوز الفريق في المباراة القادمة؟
- **Feature Engineering:** أصنع أعمدة جديدة من الموجودة، مثل `Goals_Per_Match` (عدد الأهداف ÷ عدد المباريات)، أو `Win_Rate_Last5` (نسبة الفوز في آخر 5 مباريات).
- **Feature Selection:** أختار أي الأعمدة تدخل الموديل. أحذف `Match_ID` لأنه رقم تعريفي لا يفيد التنبؤ، وأحذف `Final_Score` لأنه نتيجة المباراة نفسها.

### 2. الفرق بين Filter و Wrapper و Embedded

| الطريقة | الفكرة | مثال | ميزة | عيب |
|---|---|---|---|---|
| Filter | أقيّم كل عمود بالأرقام (ارتباط، تباين) قبل أي موديل | `corr()` مع الـ Target | سريعة وبسيطة | لا ترى تفاعل الأعمدة مع بعضها |
| Wrapper | أجرّب مجموعات أعمدة بموديل حقيقي وأقارن النتائج | إزالة عمود عمود (RFE) | أدق لأنها تقيس الأداء الفعلي | بطيئة |
| Embedded | الموديل نفسه يختار الأعمدة أثناء التدريب | Lasso أو أهمية الأعمدة في Random Forest | توازن بين السرعة والدقة | مرتبطة بنوع موديل معين |

نستخدم **Filter فقط** هذا الأسبوع لأنه لا يحتاج تدريب موديل.

### 3. Data Leakage
- `Disbursement_Date`: **لا أستخدمه**. يُسجَّل بعد الموافقة، فيسرّب الإجابة للموديل.
- `Manager_Final_Decision`: **لا أستخدمه**. هو القرار نفسه تقريبًا، فالموديل ينسخ الجواب.
- `Customer_Since_Years`: **أستخدمه**. معلومة معروفة قبل طلب القرض.

القاعدة: العمود مقبول إذا كان معروفًا وقت اتخاذ القرار، وغير مقبول إذا نشأ بعده.

### 4. «ارتباط قريب من الصفر يعني العمود لا يفيد»
ليس صحيحًا دائمًا، لأن `corr()` يقيس العلاقة الخطية فقط. مثال من الرياضة: ساعات التدريب. قليلها لا يفيد، والمعتدل يرفع المستوى، والمفرط يسبب إصابة. العلاقة على شكل قوس وليست خطًا مستقيمًا، فالارتباط يطلع قريبًا من الصفر مع أن للعمود أثرًا حقيقيًا. لذلك لا أحذف عمودًا بالارتباط وحده.

##Task 2


In [67]:
import pandas as pd

# قراءة الملف (المسار "/loan_clean.csv" لأن الملف في الجذر عندك، لو رفعتيه داخل content خليها "loan_clean.csv")
df = pd.read_csv("/loan_clean.csv")
print(df.shape)

# 1. Total_Income
df["Total_Income"] = df["ApplicantIncome"] + df["CoapplicantIncome"]

# 2. Has_Coapplicant
df["Has_Coapplicant"] = (df["CoapplicantIncome"] > 0).astype(int)

# 3. Loan_to_Income (LoanAmount بالآلاف)
df["Loan_to_Income"] = (df["LoanAmount"] * 1000) / df["Total_Income"]

# 4. Monthly_Payment و Payment_to_Income
df["Monthly_Payment"] = (df["LoanAmount"] * 1000) / df["Loan_Amount_Term"]
df["Payment_to_Income"] = df["Monthly_Payment"] / df["Total_Income"]

# 5. أول 5 صفوف + describe للنسبتين
new_cols = ["Total_Income", "Has_Coapplicant", "Loan_to_Income", "Monthly_Payment", "Payment_to_Income"]
print(df[new_cols].head())
print(df[["Loan_to_Income", "Payment_to_Income"]].describe())

# 6. تقسيم إلى 4 فئات بـ qcut، ونسبة الموافقة في كل فئة
df["Payment_Group"] = pd.qcut(df["Payment_to_Income"], 4, labels=["Q1", "Q2", "Q3", "Q4"])
df["Income_Group"] = pd.qcut(df["ApplicantIncome"], 4, labels=["Q1", "Q2", "Q3", "Q4"])

print(df.groupby("Payment_Group", observed=True)["Loan_Status"].mean().round(3))
print(df.groupby("Income_Group", observed=True)["Loan_Status"].mean().round(3))
print(df.groupby("Has_Coapplicant")["Loan_Status"].mean().round(3))

# 8. حفظ الملف
df.to_csv("loan_features.csv", index=False)
print("انحفظ:", df.shape)

(614, 13)
   Total_Income  Has_Coapplicant  Loan_to_Income  Monthly_Payment  \
0        5849.0                0       21.884083       355.555556   
1        6091.0                1       21.014612       355.555556   
2        3000.0                0       22.000000       183.333333   
3        4941.0                1       24.286582       333.333333   
4        6000.0                0       23.500000       391.666667   

   Payment_to_Income  
0           0.060789  
1           0.058374  
2           0.061111  
3           0.067463  
4           0.065278  
       Loan_to_Income  Payment_to_Income
count      614.000000         614.000000
mean        23.876787           0.077391
std          8.707161           0.079017
min          2.522916           0.007008
25%         19.220552           0.055302
50%         24.129209           0.068970
75%         28.158020           0.081695
max         82.711645           1.236873
Payment_Group
Q1    0.643
Q2    0.712
Q3    0.739
Q4    0.656
Name: 

### 5. القيم الغريبة
- أعلى `Payment_to_Income` هو 1.24، أي أن القسط الشهري أكبر من الدخل الكلي، وهذه قيمة غريبة. والمتوسط 0.077 والوسيط 0.069، فهي حالة نادرة.
- أعلى `Loan_to_Income` هو 82.7 مقابل متوسط 23.9، فهو بعيد عن البقية.

### 7. ملاحظات
- **Has_Coapplicant غيّرت نسبة الموافقة بوضوح:** 64.8% بدون شريك مقابل 71.8% مع شريك. وهذا كما توقعت، لأن دخل الشريك يرفع القدرة على السداد.
- **Payment_to_Income غيّرت النسبة، لكن ليس بشكل مستقيم:** Q1 64.3%، Q2 71.2%، Q3 73.9%، Q4 65.6%. توقعت أن ترتفع الموافقة كلما قلّ القسط، لكن الفئة الأقل قسطًا (Q1) أقلهم موافقة. هذا فاجأني.
- **ApplicantIncome لم تغيّر شيئًا تقريبًا:** كل الفئات بين 68.2% و69.3%. توقعت أن الدخل الأعلى يزيد الموافقة، فكان توقعي خاطئًا هنا.
- الخلاصة: Feature الشريك والقسط أفادا أكثر من دخل المتقدم وحده.

##Task 3

In [68]:
import numpy as np

# 1. المعرّفات والأعمدة الثابتة: nunique لكل عمود
print(df.nunique())
print("أعمدة قيمتها واحدة:", [c for c in df.columns if df[c].nunique() == 1])

# 2. الأعمدة التصنيفية: نسبة الموافقة في كل مجموعة، والفجوة
cat_cols = ["Gender", "Married", "Education", "Self_Employed", "Property_Area", "Dependents"]
rows = []
for col in cat_cols:
    rates = df.groupby(col)["Loan_Status"].mean()
    rows.append({"column": col, "highest": rates.idxmax(), "highest_rate": round(rates.max(), 3),
                 "lowest": rates.idxmin(), "lowest_rate": round(rates.min(), 3),
                 "gap": round(rates.max() - rates.min(), 3)})
gaps = pd.DataFrame(rows).sort_values("gap", ascending=False)
print(gaps)

# 3. الأعمدة الرقمية: corr مع Target مرتبة
num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term",
            "Credit_History", "Total_Income", "Has_Coapplicant", "Loan_to_Income",
            "Monthly_Payment", "Payment_to_Income"]
corr_target = df[num_cols + ["Loan_Status"]].corr()["Loan_Status"].drop("Loan_Status")
print(corr_target.sort_values(key=abs, ascending=False).round(3))

# 4. التكرار بين الأعمدة: الارتباط بين ApplicantIncome و Total_Income
print("الارتباط:", round(df["ApplicantIncome"].corr(df["Total_Income"]), 3))

Loan_ID              614
Gender                 2
Married                2
Dependents             4
Education              2
Self_Employed          2
ApplicantIncome      505
CoapplicantIncome    287
LoanAmount           203
Loan_Amount_Term      10
Credit_History         2
Property_Area          3
Loan_Status            2
Total_Income         554
Has_Coapplicant        2
Loan_to_Income       608
Monthly_Payment      237
Payment_to_Income    611
Payment_Group          4
Income_Group           4
dtype: int64
أعمدة قيمتها واحدة: []
          column    highest  highest_rate        lowest  lowest_rate    gap
4  Property_Area  Semiurban         0.768         Rural        0.615  0.154
5     Dependents          2         0.752             1        0.647  0.105
2      Education   Graduate         0.708  Not Graduate        0.612  0.096
1        Married        Yes         0.718            No        0.629  0.089
0         Gender       Male         0.691        Female        0.670  0.022
3  Self_

### 1. المعرّفات والأعمدة الثابتة
`Loan_ID` فيه 614 قيمة مختلفة (كل صف له رقم)، فهو معرّف لا يفيد التنبؤ. لا يوجد عمود قيمته واحدة.

### 2. الأعمدة التصنيفية (الفجوة في نسبة الموافقة)
| العمود | أعلى نسبة | أقل نسبة | الفجوة |
|---|---|---|---|
| `Property_Area` | Semiurban 0.768 | Rural 0.615 | 0.154 |
| `Dependents` | 2 (0.752) | 1 (0.647) | 0.105 |
| `Education` | Graduate 0.708 | Not Graduate 0.612 | 0.096 |
| `Married` | Yes 0.718 | No 0.629 | 0.089 |
| `Gender` | Male 0.691 | Female 0.670 | 0.022 |
| `Self_Employed` | No 0.688 | Yes 0.683 | 0.005 |

### 3. الأعمدة الرقمية (الارتباط مع Target)
`Credit_History` أقوى عمود بفارق كبير (0.541). بعده `Loan_to_Income` (-0.086) و`Has_Coapplicant` (0.075)، والباقي قريب من الصفر، وأضعفها `ApplicantIncome` (-0.005).

### 4. التكرار بين الأعمدة
الارتباط بين `ApplicantIncome` و`Total_Income` هو 0.893، وهو عالٍ جدًا، فنحتاج واحدًا فقط. أبقي `Total_Income` لأنه يشمل دخل الشريك، وأحذف `ApplicantIncome`.

### 5. جدول القرار

| العمود | القرار | الدليل (رقم) | السبب |
|---|---|---|---|
| `Loan_ID` | أحذف | 614 قيمة مختلفة | معرّف لا يفيد |
| `Credit_History` | أبقي | ارتباط 0.541 | أقوى عمود |
| `Property_Area` | أبقي | فجوة 0.154 | أكبر فجوة بين التصنيفية |
| `Dependents` | أبقي | فجوة 0.105 | فرق واضح |
| `Education` | أبقي | فجوة 0.096 | فرق واضح |
| `Married` | أبقي | فجوة 0.089 | فرق واضح |
| `Has_Coapplicant` | أبقي | ارتباط 0.075، وموافقة 71.8% مقابل 64.8% | يغيّر النتيجة |
| `Loan_to_Income` | أبقي | ارتباط -0.086 | أعلى ارتباط بعد السجل الائتماني |
| `Payment_to_Income` | أبقي | ارتباط -0.051 | يعبّر عن القدرة على السداد |
| `Total_Income` | أبقي | ارتباط 0.893 مع ApplicantIncome | يغني عن ApplicantIncome |
| `ApplicantIncome` | أحذف | تكرار مع Total_Income | نفس المعلومة |
| `CoapplicantIncome` | أحذف | تكرار جزئي (داخل Total_Income) | موجود ضمنه |
| `LoanAmount` | أحذف | ارتباط -0.033 | ضعيف، ومعبَّر عنه في Loan_to_Income |
| `Loan_Amount_Term` | أحذف | ارتباط -0.023 | ضعيف |
| `Monthly_Payment` | أحذف | ارتباط -0.012 | ضعيف، ومعبَّر عنه في Payment_to_Income |
| `Self_Employed` | أحذف | فجوة 0.005 | لا يفرّق |
| `Gender` | أحذف | فجوة 0.022 | ضعيف، وغير عادل (انظري تحت) |

**Gender والعدل:** الفجوة 0.022 صغيرة، فالعمود لا يضيف قدرة تنبؤية. وحتى لو أضاف، استخدام الجنس في قرار قرض تمييز غير عادل. فأحذفه بسبب الدليل (ضعيف) وبسبب العدل معًا.

`Payment_Group` و`Income_Group` أنشأناهما لمقارنة النسب فقط، فلا أدخلهما الموديل.

##Task 5


In [69]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 300
life = rng.integers(0, 2, n)                       # 1 = signs of life found
probes = pd.DataFrame({
    "probe_id": [f"P{i:03d}" for i in range(n)],
    "mission_code": "AX-7",
    "methane_ppm": np.round(5 + life * 12 + rng.normal(0, 3, n), 1),
    "temperature_c": np.round(-40 + life * 25 + rng.normal(0, 15, n), 1),
    "radiation": np.round(rng.normal(100, 10, n), 1),
    "humidity": np.round(20 + life * 6 + rng.normal(0, 8, n), 1),
    "life_detected": life,
})
probes["temperature_f"] = probes["temperature_c"] * 9 / 5 + 32
print(probes.shape)
print(probes.head())

(300, 8)
  probe_id mission_code  methane_ppm  temperature_c  radiation  humidity  \
0     P000         AX-7         17.7          -18.0       71.4      19.2   
1     P001         AX-7          2.2          -61.9      108.9      27.9   
2     P002         AX-7          9.7          -37.4      102.3      24.8   
3     P003         AX-7         21.6          -24.5      119.1      20.4   
4     P004         AX-7          6.7          -54.5      101.9      14.7   

   life_detected  temperature_f  
0              1          -0.40  
1              0         -79.42  
2              0         -35.32  
3              1         -12.10  
4              0         -66.10  


In [70]:
# 1. أي عمود معرّف؟ وأي عمود ثابت؟
print(probes.nunique())
print("أعمدة قيمتها واحدة:", [c for c in probes.columns if probes[c].nunique() == 1])

# 2. الارتباط بين كل حساس و life_detected، مرتبًا
sensors = ["methane_ppm", "temperature_c", "radiation", "humidity", "temperature_f"]
corr_life = probes[sensors + ["life_detected"]].corr()["life_detected"].drop("life_detected")
print(corr_life.sort_values(key=abs, ascending=False).round(3))

probe_id         300
mission_code       1
methane_ppm      160
temperature_c    246
radiation        212
humidity         192
life_detected      2
temperature_f    246
dtype: int64
أعمدة قيمتها واحدة: ['mission_code']
methane_ppm      0.882
temperature_f    0.643
temperature_c    0.643
humidity         0.397
radiation        0.110
Name: life_detected, dtype: float64


## Task 5

### 1. قواعد Filter
- `probe_id` معرّف (300 قيمة مختلفة)، أحذفه.
- `mission_code` ثابت (قيمة واحدة "AX-7")، أحذفه.

### 2. الارتباط مع life_detected
| الحساس | الارتباط |
|---|---|
| `methane_ppm` | 0.882 |
| `temperature_c` | 0.643 |
| `temperature_f` | 0.643 |
| `humidity` | 0.397 |
| `radiation` | 0.110 |

### 3. حساسان يقيسان الشيء نفسه
`temperature_c` و`temperature_f` يقيسان الحرارة نفسها بوحدتين مختلفتين، وارتباطهما مع الهدف متطابق (0.643). أبقي `temperature_c` وأحذف `temperature_f` لأنه نسخة منه ولا يضيف معلومة.

## Task 5

### 1. قواعد Filter
- `probe_id` معرّف (300 قيمة مختلفة)، أحذفه.
- `mission_code` ثابت (قيمة واحدة "AX-7")، أحذفه.

### 2. الارتباط مع life_detected
| الحساس | الارتباط |
|---|---|
| `methane_ppm` | 0.882 |
| `temperature_c` | 0.643 |
| `temperature_f` | 0.643 |
| `humidity` | 0.397 |
| `radiation` | 0.110 |

### 3. حساسان يقيسان الشيء نفسه
`temperature_c` و`temperature_f` يقيسان الحرارة نفسها بوحدتين مختلفتين، وارتباطهما مع الهدف متطابق (0.643). أبقي `temperature_c` وأحذف `temperature_f` لأنه نسخة منه ولا يضيف معلومة.

In [71]:
# 3. تأكيد أن الحساسين متطابقان
print("الارتباط بينهما:", round(probes["temperature_c"].corr(probes["temperature_f"]), 3))

# 4. متوسط كل حساس بين الكواكب
print(probes.groupby("life_detected")[["methane_ppm", "temperature_c", "radiation", "humidity"]].mean().round(2))

الارتباط بينهما: 1.0
               methane_ppm  temperature_c  radiation  humidity
life_detected                                                 
0                     5.38         -40.31      98.74     19.14
1                    17.04         -14.78     100.96     25.63


### 4. مقارنة المتوسطات (بين الكواكب التي فيها حياة والتي ليس فيها)

| الحساس | بلا حياة (0) | فيها حياة (1) | الفرق |
|---|---|---|---|
| `methane_ppm` | 5.38 | 17.04 | 11.66 |
| `temperature_c` | -40.31 | -14.78 | 25.53 |
| `humidity` | 19.14 | 25.63 | 6.49 |
| `radiation` | 98.74 | 100.96 | 2.22 |

الميثان والحرارة فرقهما واضح جدًا، والرطوبة فرقها متوسط، والإشعاع شبه متساوٍ في المجموعتين، فلا يفرّق بين الكواكب.

##5. القرار: الحساسات الثلاثة للمسبار القادم

| الترتيب | الحساس | الدليل الرقمي | السبب |
|---|---|---|---|
| 1 | `methane_ppm` | ارتباط 0.882، وفرق المتوسط 11.66 | أقوى حساس بفارق كبير |
| 2 | `temperature_c` | ارتباط 0.643، وفرق المتوسط 25.53 | تفرّق بوضوح بين المجموعتين |
| 3 | `humidity` | ارتباط 0.397، وفرق المتوسط 6.49 | أقوى من الإشعاع وتضيف معلومة |

**المستبعدة:**
- `temperature_f`: نسخة من `temperature_c` (ارتباط بينهما 1.0).
- `radiation`: ارتباط 0.110 فقط، ومتوسطاها شبه متساويين (98.74 و100.96).
- `probe_id` و`mission_code`: معرّف وعمود ثابت.

### 6. تفكير: كيف أتأكد من قراري لو عندي وقت لموديل حقيقي؟
أستخدم **Wrapper**، لأنني أجرّب مجموعات الحساسات بموديل حقيقي وأقارن أداءها. مثلًا أدرّب الموديل بالحساسات الثلاثة (الميثان والحرارة والرطوبة)، ثم أبدّل واحدًا منها بالإشعاع وأرى هل تنخفض الدقة. هذا يثبت أن اختياري صحيح بالأداء الفعلي وليس بالارتباط وحده.

ويمكن أيضًا **Embedded** مثل أهمية الأعمدة في Random Forest، فهو أسرع لأن الموديل نفسه يرتّب الحساسات أثناء التدريب. لكن Wrapper أنسب هنا لأن عدد الحساسات قليل (5)، فالتجربة سريعة، والنتيجة أوضح.

وقد يختلف قراري عن الارتباط: لو كانت العلاقة غير خطية، أو لو حساسان ضعيفان فرديًا يصبحان قويين معًا، فالارتباط لا يلتقطه والموديل قد يلتقطه.